In [ ]:
import spacy
from nltk.corpus import stopwords
import re
import os
import unicodedata
from unidecode import unidecode
import pandas as pd
from spacy.tokens import Doc

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Como o notebook está em 'corretor/src/3_avaliacao' e o .env em 'corretor',
# precisamos voltar duas pastas (../..) para achar o arquivo.
env_path = Path('../../.env')

# Carrega o arquivo explicitamente
load_dotenv(dotenv_path=env_path)

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
nlp = spacy.load("pt_core_news_lg")

In [ ]:
PATH_BASE_TESTES_FINAIS = os.getenv('PATH_BASE_TESTES_FINAIS')
PATH_GABARITO_CORRETOR = os.getenv('PATH_GABARITO_CORRETOR')
PATH_DADOS_ANONIMIZACAO_SPACY = os.getenv('PATH_DADOS_ANONIMIZACAO_SPACY')

In [ ]:
data = pd.read_csv(PATH_BASE_TESTES_FINAIS)

In [ ]:
def preprocess(relato):
    def remove_stop_words(relato):
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

In [ ]:
data["relato_tratado"] = data["relato"].apply(normalize_text).apply(preprocess)

In [ ]:
data['relato_tratado'][230]

In [ ]:
data['relato_tratado'] = data['relato_tratado'].str.lower()

In [ ]:
base_relato_teste = data.copy()

In [ ]:
def check_names(output_list : pd.DataFrame):
    gab = pd.read_csv(PATH_GABARITO_CORRETOR)
    copia_output_list = output_list.copy()
    
    copia_output_list = copia_output_list[copia_output_list['y_palavra'] != 2]
    copia_output_list['y_palavra'] = copia_output_list['y_palavra'].astype(int)

    merge_gab_output_list = copia_output_list.merge(gab[['palavra', 'y']], left_on='palavra', right_on='palavra', how='left')
    merge_gab_output_list['y'] = merge_gab_output_list['y'].fillna(0).astype(int)
    merge_gab_output_list = merge_gab_output_list[merge_gab_output_list['y_palavra'] != 2]

    return merge_gab_output_list

In [ ]:
for idx, text in base_relato_teste["relato_tratado"].items():
    resultados = []
    doc = nlp(text)
    for token in doc:
        label = 1 if token.ent_type_ == "PER" else 0
        resultados.append((token.text, label))
    
    df_resultado = pd.DataFrame(resultados, columns=["palavra", "y_palavra"])
    df_resultado['palavra'] = df_resultado['palavra'].str.upper()
    pred_gab_names = check_names(df_resultado)
    base_relato_teste.at[idx, 'spacy_qtd_tokens'] = len(df_resultado['palavra'])
    base_relato_teste.at[idx, 'spacy_qtd_pred_nomes'] = len(pred_gab_names[pred_gab_names['y_palavra'] == 1])
    base_relato_teste.at[idx, 'spacy_qtd_gab_nomes'] = len(pred_gab_names[pred_gab_names['y'] == 1])
    base_relato_teste.at[idx, 'spacy_nomes_acertados'] = len(pred_gab_names[(pred_gab_names['y_palavra'] == 1) & (pred_gab_names['y'] == 1)])
    print(f"{idx} de 230")

In [ ]:
base_relato_teste

In [ ]:
base_relato_teste.to_csv(PATH_DADOS_ANONIMIZACAO_SPACY, index=False)